# Практика · seq2seq і машинний переклад

> Лекція: [lecture.html](lecture.html) · Домашнє завдання: [homework.html](homework.html) ·
> Тест: [quiz.html](quiz.html)

> ⏱ **Зошит навчає шість мереж** — три зерна на кожну з двох конфігурацій.
> Заміряно: **близько чотирьох з половиною хвилин процесорного часу**
> на машині з чотирма ядрами й **без відеокарти**. Точне число зошит друкує сам
> в останній клітинці.

Тут рахуються **всі** числа, які називає лекція, і в тому самому порядку.

Що зробимо:

1. Витягнемо з корпусу **паралельні пари** «англійський оригінал → український переклад».
2. Складемо енкодер і декодер із двох `nn.GRU` і перевіримо рукописний крок декодера
   проти бібліотечного.
3. Навчимо модель **подаванням еталона** (`teacher forcing`) і заміряємо частку
   правильних токенів — по трьох зернах.
4. Розберемо цю частку **за довжиною джерела** й побачимо горб.
5. Увімкнемо **справжню генерацію** без еталона й побачимо, наскільки все гірше.
6. Порівняємо жадібне декодування з `beam search` різної ширини.
7. Напишемо **власний BLEU** і звіримо його з реалізацією `nltk`.
8. Заміряємо, чи справді винен розмір вектора контексту: та сама модель із `H = 64`.

## 1 · Середовище

Перша клітинка друкує версії й просить числові бібліотеки рахувати **в один потік**.

Це не оптимізація, а умова того, щоб замір часу щось означав. На зайнятій машині
чотири потоки більшу частину часу чекають одне на одного — і це очікування
записується в процесорний час як робота. Змінні мусять стояти **до** імпорту numpy:
пізніше вони вже не подіють.

In [ ]:
import os
# Просимо numpy й torch рахувати в один потік.
# Рядки мусять стояти ДО імпорту numpy — інакше бібліотека вже прочитала своє.
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

import sys, re, math, glob, gettext, time, random, itertools, collections
import numpy as np
import torch
import torch.nn as nn
import nltk
torch.set_num_threads(1)

NOTEBOOK_START = time.process_time()   # звідси рахуємо загальний час зошита

print("Python  ", sys.version.split()[0])
print("numpy   ", np.__version__)
print("torch   ", torch.__version__)
print("nltk    ", nltk.__version__)
print("потоків :", os.environ["OMP_NUM_THREADS"], "· torch:", torch.get_num_threads())

## 2 · Паралельний корпус: переклад, який уже лежить у системі

Корпус той самий, що в усьому курсі, але цього разу нас цікавить його друга
властивість. У `.mo`-файлах української локалі кожен запис — це **пара**:
англійський рядок інтерфейсу і його український переклад. Це і є **паралельний
корпус**, тобто той самий зміст двома мовами, вирівняний по реченнях.

Саме на такому матеріалі й навчають машинний переклад. Нам не треба нічого
завантажувати: пари вже лежать на диску.

Якщо української локалі на машині немає, вмикається запасний **синтетичний**
корпус із шаблонів. Він друкує гучне попередження: числа на ньому будуть **інші**,
і посилатися на них не можна. Запасний шлях прогнано зошитом наскрізь — він
справді працює, а не просто існує.

In [ ]:
def load_system_pairs():
    """Читаємо .mo-файли української локалі.
    Повертаємо пари (англійський оригінал, український переклад)."""
    pairs = []
    for path in sorted(glob.glob('/usr/share/locale/uk/LC_MESSAGES/*.mo')):
        try:
            with open(path, 'rb') as f:
                catalog = gettext.GNUTranslations(f)
        except Exception:
            continue                      # зламаний або чужий формат — пропускаємо
        for source, target in catalog._catalog.items():
            # службовий заголовок каталогу має ключ '' і текстом не є
            if isinstance(source, str) and isinstance(target, str) \
               and len(target) > 30 and 'Project-Id' not in target:
                pairs.append((source, target))
    return pairs


# Запасний корпус: шаблони інтерфейсних фраз. Потрібен лише тоді,
# коли української локалі на машині немає взагалі.
FALLBACK_VERBS = [("open", "відкрити"), ("save", "зберегти"), ("delete", "видалити"),
                  ("copy", "скопіювати"), ("move", "перемістити"), ("rename", "перейменувати"),
                  ("create", "створити"), ("read", "прочитати"), ("write", "записати"),
                  ("close", "закрити"), ("load", "завантажити"), ("send", "надіслати")]
FALLBACK_NOUNS = [("file", "файл"), ("folder", "теку"), ("document", "документ"),
                  ("archive", "архів"), ("report", "звіт"), ("message", "повідомлення"),
                  ("image", "зображення"), ("table", "таблицю"), ("list", "список"),
                  ("account", "обліковий запис"), ("settings", "налаштування"), ("key", "ключ")]
FALLBACK_ADJS = [("", ""), ("new ", "новий "), ("old ", "старий "),
                 ("selected ", "вибраний "), ("current ", "поточний "), ("temporary ", "тимчасовий ")]
FALLBACK_TEMPLATES = [
    ("{n} missing", "{n} відсутній"),
    ("{n} not found", "{n} не знайдено"),
    ("{v} the {a}{n}", "{v} {a}{n}"),
    ("cannot {v} the {a}{n}", "не вдалося {v} {a}{n}"),
    ("failed to {v} the {a}{n}", "не вдалося {v} {a}{n}"),
    ("do you want to {v} the {a}{n}", "чи хочете ви {v} {a}{n}"),
    ("click here to {v} the {a}{n}", "натисніть тут щоб {v} {a}{n}"),
    ("the {a}{n} was not found", "{a}{n} не знайдено"),
    ("please {v} the {a}{n} first", "спершу {v} {a}{n}"),
    ("unable to {v} this {n}", "неможливо {v} цей {n}"),
    ("the {n} is already open", "{n} вже відкрито"),
]

def fallback_pairs():
    """Синтетичні пари з шаблонів — на випадок, коли локалі немає."""
    out = []
    for (verb_en, verb_uk), (noun_en, noun_uk), (adj_en, adj_uk), (t_en, t_uk) in \
            itertools.product(FALLBACK_VERBS, FALLBACK_NOUNS, FALLBACK_ADJS, FALLBACK_TEMPLATES):
        out.append((t_en.format(v=verb_en, n=noun_en, a=adj_en),
                    t_uk.format(v=verb_uk, n=noun_uk, a=adj_uk)))
    # частина шаблонів не використовує прикметник, тож однакові пари повторюються
    out = list(dict.fromkeys(out))
    random.Random(0).shuffle(out)
    return out


raw_pairs = load_system_pairs()
REAL_CORPUS = len(raw_pairs) > 20000
if not REAL_CORPUS:
    print("⚠️ УВАГА: української локалі на машині немає або в ній замало рядків.")
    print("⚠️ Вмикаю СИНТЕТИЧНИЙ корпус. Усі числа нижче будуть ІНШІ,")
    print("⚠️ і цитувати їх разом із лекцією не можна.")
    raw_pairs = fallback_pairs()

print("джерело :", "системна локаль" if REAL_CORPUS else "СИНТЕТИЧНИЙ запасний корпус")
print("записів :", len(raw_pairs))
print()
for english, ukrainian in raw_pairs[:3]:
    print(" ", english[:58])
    print("  →", ukrainian[:58])

## 3 · Пари, у яких обидва боки короткі

Токенізатор для української — канон курсу: букви абетки, апостроф усередині слова
не рве його надвоє. Для англійської — та сама ідея латиницею.

Далі беремо лише ті пари, де **обидва** боки мають від 2 до 8 слів. Причин дві.
Довгі речення дорожчі за часом, і їх у нас усе одно небагато; а короткі пари з
одного слова майже завжди — назва пункту меню, з якої вчитися нема чого.

In [ ]:
# Канонічні токенізатори курсу. Апостроф — звʼязка ВСЕРЕДИНІ слова, а не символ слова.
UK_TOKEN = re.compile(r"[а-яїієґ]+(?:['ʼ’][а-яїієґ]+)*")
EN_TOKEN = re.compile(r"[a-z]+(?:'[a-z]+)*")

MIN_WORDS, MAX_WORDS = 2, 8

pairs = []
for english, ukrainian in raw_pairs:
    src = EN_TOKEN.findall(english.lower())
    tgt = UK_TOKEN.findall(ukrainian.lower())
    if MIN_WORDS <= len(src) <= MAX_WORDS and MIN_WORDS <= len(tgt) <= MAX_WORDS:
        pairs.append((src, tgt))

by_source_length = collections.Counter(len(src) for src, tgt in pairs)

print("усього записів    :", len(raw_pairs))
print("пар 2-8 слів з обох боків:", len(pairs))
print()
print("скільки пар якої довжини джерела:")
for length in range(MIN_WORDS, MAX_WORDS + 1):
    print(f"  {length} слів: {by_source_length[length]:6d}")

Подивимось на кілька пар підряд — **не вибраних**, а просто перших зі списку.
Це важлива звичка: щойно починаєш вибирати приклади, демонстрація перестає
щось доводити.

In [ ]:
for src, tgt in pairs[:6]:
    print(f"{' '.join(src):<42} → {' '.join(tgt)}")
print()
print("Одразу видно дві речі. Перша: це переклади інтерфейсів, а не художній текст.")
print("Друга: підстановки на кшталт %s токенізатор ріже, і від них лишається")
print("самотня літера s. Це шум, з яким модель житиме.")

## 4 · Поділ і два словники

Поділ 90/10 із зерном 0 — той самий, що в усьому блоці. Зерно поділу **не** міняємо
разом із зерном моделі: інакше три прогони відрізнялись би і даними, і ініціалізацією,
і ми не знали б, що саме дало розкид.

Словників тут **два**: один для англійського боку, другий для українського. Це перша
відмінність seq2seq від усього, що ми робили досі: вхід і вихід моделі живуть у
різних алфавітах.

Чотири спецтокени:

| | |
|---|---|
| `<pad>` | добивання коротких речень у батчі до однакової довжини |
| `<bos>` | «почни писати» — перший вхід декодера |
| `<eos>` | «я закінчив» — без нього декодер не знає, коли спинитись |
| `<unk>` | слово, якого немає в словнику |

In [ ]:
PAD, BOS, EOS, UNK = 0, 1, 2, 3
SPECIAL = ['<pad>', '<bos>', '<eos>', '<unk>']

SPLIT_SEED = 0          # зерно ПОДІЛУ даних — одне на всі прогони
TRAIN_PAIRS = 18000   # скільки навчальних пар беремо (бюджет часу)
MIN_COUNT = 3           # слово рідше за це не потрапляє в словник

order = np.random.default_rng(SPLIT_SEED).permutation(len(pairs))
cut = int(0.9 * len(pairs))
train_all = [pairs[i] for i in order[:cut]]
valid = [pairs[i] for i in order[cut:]]
train = train_all[:TRAIN_PAIRS]

def build_vocab(sequences, min_count):
    """Словник за навчальною частиною. Спецтокени завжди перші й у сталому порядку."""
    counts = collections.Counter(word for seq in sequences for word in seq)
    words = sorted(word for word, n in counts.items() if n >= min_count)
    itos = SPECIAL + words
    return {word: i for i, word in enumerate(itos)}, itos

src_stoi, src_itos = build_vocab([src for src, tgt in train], MIN_COUNT)
tgt_stoi, tgt_itos = build_vocab([tgt for src, tgt in train], MIN_COUNT)

# скільки токенів перевірної частини словник не знає
valid_tgt_tokens = [word for src, tgt in valid for word in tgt]
unknown_share = sum(1 for w in valid_tgt_tokens if w not in tgt_stoi) / len(valid_tgt_tokens)

print("навчальних пар     :", len(train), f"(з {len(train_all)} доступних)")
print("перевірних пар     :", len(valid))
print("англійський словник:", len(src_itos))
print("український словник:", len(tgt_itos))
print(f"частка <unk> серед українських токенів перевірної частини: {unknown_share:.4f}")

## 5 · Модель: енкодер, вектор контексту, декодер

Уся конструкція вміщається в один клас і три рядки думки.

**Енкодер** читає англійське речення по слову. Це звичайна GRU з теми 18: на кожному
кроці вона оновлює свій стан. Коли слова скінчились, ми беремо **останній стан** —
і оголошуємо його поданням усього речення. Це і є **вектор контексту**: `H` чисел,
і їх рівно стільки ж для речення з двох слів і для речення з восьми.

**Декодер** — теж GRU, але українська. Її стартовий стан — той самий вектор контексту.
Далі вона пише по слову: бере попереднє слово, оновлює стан, лінійний шар перетворює
стан на бали для кожного слова українського словника.

Між ними немає нічого. Жодного іншого каналу, яким англійське речення могло б
дійти до декодера, крім цих `H` чисел.

In [ ]:
EMB = 128     # довжина вектора слова
HID = 192     # довжина стану GRU — вона ж довжина вектора контексту
BATCH = 128
EPOCHS = 2
LR = 2e-3

class Seq2Seq(nn.Module):
    def __init__(self, n_src, n_tgt, emb=EMB, hid=HID):
        super().__init__()
        self.src_emb = nn.Embedding(n_src, emb, padding_idx=PAD)
        self.encoder = nn.GRU(emb, hid, batch_first=True)
        self.tgt_emb = nn.Embedding(n_tgt, emb, padding_idx=PAD)
        self.decoder = nn.GRU(emb, hid, batch_first=True)
        self.out = nn.Linear(hid, n_tgt)

    def context(self, src, src_len):
        """Стискаємо все вхідне речення в один вектор — стан GRU після останнього слова.
        pack_padded_sequence потрібен, щоб добивання <pad> не потрапило в стан."""
        packed = nn.utils.rnn.pack_padded_sequence(
            self.src_emb(src), src_len, batch_first=True, enforce_sorted=False)
        _, hidden = self.encoder(packed)
        return hidden                       # (1, батч, HID)

    def forward(self, src, src_len, tgt_in):
        """Навчальний прохід: декодерові подають еталонний префікс цілком."""
        hidden = self.context(src, src_len)
        output, _ = self.decoder(self.tgt_emb(tgt_in), hidden)
        return self.out(output)


model_shape = Seq2Seq(len(src_itos), len(tgt_itos))
params = sum(p.numel() for p in model_shape.parameters())
print("довжина вектора слова   :", EMB)
print("довжина вектора контексту:", HID)
print("параметрів усього       :", f"{params:,}".replace(",", " "))
print()
print("на що вони витрачені:")
for name, block in [("ембединги англійські", model_shape.src_emb),
                    ("енкодер", model_shape.encoder),
                    ("ембединги українські", model_shape.tgt_emb),
                    ("декодер", model_shape.decoder),
                    ("вихідний шар", model_shape.out)]:
    n = sum(p.numel() for p in block.parameters())
    print(f"  {name:<24} {n:>9,}".replace(",", " "), f"  {100*n/params:5.1f} %")

### Перевірка: усередині `nn.GRU` немає магії

Бібліотечний виклик обробляє всю послідовність одним рядком. Напишемо той самий
крок руками — за формулою GRU з теми 18 — і звіримо результат із бібліотечним.

Якщо два числа збігаються, значить рекурентність — це справді цикл із трьох
матричних множень, а не чорна скринька.

In [ ]:
def gru_step_by_hand(gru, x_t, h_prev):
    """Один крок GRU за формулою. Ваги беремо з бібліотечного шару."""
    w_ih, w_hh = gru.weight_ih_l0, gru.weight_hh_l0
    b_ih, b_hh = gru.bias_ih_l0, gru.bias_hh_l0
    hid = gru.hidden_size
    # у torch три брами лежать одним стосом: скидання, оновлення, кандидат
    gates_x = x_t @ w_ih.T + b_ih
    gates_h = h_prev @ w_hh.T + b_hh
    reset = torch.sigmoid(gates_x[:, :hid] + gates_h[:, :hid])
    update = torch.sigmoid(gates_x[:, hid:2*hid] + gates_h[:, hid:2*hid])
    # кандидат бачить попередній стан ЧЕРЕЗ браму скидання
    candidate = torch.tanh(gates_x[:, 2*hid:] + reset * gates_h[:, 2*hid:])
    return (1 - update) * candidate + update * h_prev

torch.manual_seed(0)
toy = Seq2Seq(len(src_itos), len(tgt_itos))
toy_input = toy.tgt_emb(torch.tensor([[5, 7, 9]]))          # три слова
with torch.no_grad():
    start = torch.zeros(1, 1, HID)
    library_output, _ = toy.decoder(toy_input, start)
    hand = start[0]
    hand_steps = []
    for step in range(toy_input.shape[1]):
        hand = gru_step_by_hand(toy.decoder, toy_input[:, step, :], hand)
        hand_steps.append(hand)
    hand_output = torch.stack(hand_steps, dim=1)

gap = (library_output - hand_output).abs().max().item()
print(f"найбільша розбіжність між рукописним і бібліотечним кроком: {gap:.3e}")
assert torch.allclose(library_output, hand_output, atol=1e-5), "розрахунок розійшовся!"
print("✅ збігається — усередині nn.GRU той самий цикл")

## 6 · Навчання: подавання еталона

Декодер має вчитися писати переклад по слову. Але якщо на другому кроці подати йому
те, що він сам щойно написав, а написав він дурницю, — далі все навчання піде
на виправлення власного сміття.

Тому декодерові подають **еталон**: на кожному кроці на вхід іде правильне попереднє
слово з людського перекладу, а не власне передбачення. Це і є `teacher forcing`,
українською — **подавання еталона**.

Технічно це один рядок: вхід декодера — еталонний переклад із `<bos>` спереду,
а очікуваний вихід — той самий переклад, зсунутий на одну позицію, з `<eos>` у кінці.

In [ ]:
def encode_seq(words, stoi, add_bos=False, add_eos=True):
    """Слова → номери. <unk> для позасловникових, <eos> у кінці."""
    ids = [stoi.get(word, UNK) for word in words]
    if add_bos:
        ids = [BOS] + ids
    if add_eos:
        ids = ids + [EOS]
    return ids

def pad_batch(sequences):
    """Добиваємо короткі речення нулями (<pad>) до довжини найдовшого в батчі."""
    width = max(len(s) for s in sequences)
    block = np.zeros((len(sequences), width), dtype=np.int64)
    for i, seq in enumerate(sequences):
        block[i, :len(seq)] = seq
    return torch.from_numpy(block)

# один раз переводимо всі пари в номери
train_ids = [(encode_seq(src, src_stoi), encode_seq(tgt, tgt_stoi, add_bos=True))
             for src, tgt in train]
valid_ids = [(encode_seq(src, src_stoi), encode_seq(tgt, tgt_stoi, add_bos=True), len(src))
             for src, tgt in valid]

example_src, example_tgt = train_ids[0]
print("англійське джерело :", [src_itos[i] for i in example_src])
print("вхід декодера      :", [tgt_itos[i] for i in example_tgt[:-1]])
print("що він має вгадати :", [tgt_itos[i] for i in example_tgt[1:]])
print()
print("Тобто на кожному кроці декодер бачить ПРАВИЛЬНЕ попереднє слово")
print("і мусить назвати наступне.")

In [ ]:
def train_model(seed, hid=HID, epochs=EPOCHS, data=None):
    """Навчання однієї моделі. Повертає модель і процесорний час навчання."""
    data = train_ids if data is None else data
    torch.manual_seed(seed)
    np.random.seed(seed)
    model = Seq2Seq(len(src_itos), len(tgt_itos), hid=hid)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    # <pad> у цілі не карається: це добивання, а не слово
    loss_fn = nn.CrossEntropyLoss(ignore_index=PAD)
    started = time.process_time()
    last_loss = float('nan')
    for epoch in range(epochs):
        shuffled = np.random.permutation(len(data))
        for start in range(0, len(data), BATCH):
            chunk = shuffled[start:start + BATCH]
            sources = [data[i][0] for i in chunk]
            targets = [data[i][1] for i in chunk]
            src = pad_batch(sources)
            src_len = torch.tensor([len(s) for s in sources])
            tgt = pad_batch(targets)
            logits = model(src, src_len, tgt[:, :-1])       # подаємо еталонний префікс
            loss = loss_fn(logits.reshape(-1, logits.size(-1)), tgt[:, 1:].reshape(-1))
            optimizer.zero_grad()
            loss.backward()
            # без обрізання норми рекурентна мережа зривається на рідкісному батчі
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            last_loss = loss.item()
    return model, time.process_time() - started, last_loss

print("функцію навчання визначено; далі — три зерна")

### Три зерна

Зерно міняє **ініціалізацію ваг і порядок батчів**, але не дані: поділ фіксовано
зерном 0 вище. Три прогони покажуть, який розкид дає сам випадок — і будь-яка
різниця, менша за цей розкид, різницею не є.

In [ ]:
SEEDS = (0, 1, 2)

def ratio(hits, total):
    """Частка з запобіжником: якщо прикладів такої довжини немає, ділити нема на що."""
    return hits / total if total else float('nan')

def token_accuracy(model, data=None):
    """Частка правильно вгаданих токенів ПРИ ПОДАВАННІ ЕТАЛОНА,
    окремо по кожній довжині джерела. <pad> не рахуємо."""
    data = valid_ids if data is None else data
    model.eval()
    correct = np.zeros(MAX_WORDS + 1)
    total = np.zeros(MAX_WORDS + 1)
    with torch.no_grad():
        for start in range(0, len(data), 256):
            chunk = data[start:start + 256]
            sources = [item[0] for item in chunk]
            src = pad_batch(sources)
            src_len = torch.tensor([len(s) for s in sources])
            tgt = pad_batch([item[1] for item in chunk])
            predicted = model(src, src_len, tgt[:, :-1]).argmax(-1)
            gold = tgt[:, 1:]
            mask = gold != PAD
            hits = ((predicted == gold) & mask).sum(1).numpy()
            counts = mask.sum(1).numpy()
            for i, item in enumerate(chunk):
                correct[item[2]] += hits[i]
                total[item[2]] += counts[i]
    return correct, total

models, train_times, accuracies, per_length = [], [], [], []
for seed in SEEDS:
    model, seconds, loss = train_model(seed)
    correct, total = token_accuracy(model)
    models.append(model)
    train_times.append(seconds)
    accuracies.append(correct.sum() / total.sum())
    per_length.append([ratio(correct[L], total[L]) for L in range(MIN_WORDS, MAX_WORDS + 1)])
    print(f"зерно {seed}: частка правильних {accuracies[-1]:.4f} · "
          f"навчання {seconds:.1f} с · остання втрата {loss:.4f}")

def spread(values):
    """Півширина купи: половина відстані між найменшим і найбільшим."""
    return (max(values) - min(values)) / 2

print()
print(f"РАЗОМ: частка правильних токенів {np.mean(accuracies):.4f} ±{spread(accuracies):.4f}")
print(f"       навчання {np.mean(train_times):.1f} ±{spread(train_times):.1f} с на зерно")

## 7 · Частка правильних за довжиною джерела

Тепер головне число теми. Розкладемо ту саму частку за довжиною **англійського**
речення — від двох слів до восьми.

In [ ]:
per_length = np.array(per_length)                 # (зерно, довжина)
mean_by_length = per_length.mean(axis=0)
spread_by_length = (per_length.max(axis=0) - per_length.min(axis=0)) / 2
valid_by_length = collections.Counter(item[2] for item in valid_ids)

print("по зернах, щоб розкид можна було перевірити руками:")
for j, seed in enumerate(SEEDS):
    print(f"  зерно {seed}: " + ' '.join(f'{v:.4f}' for v in per_length[j]))
print()
print(f"{'слів':>5} {'пар':>7} {'правильних':>12} {'розкид':>9}")
for i, length in enumerate(range(MIN_WORDS, MAX_WORDS + 1)):
    print(f"{length:>5} {valid_by_length[length]:>7} "
          f"{mean_by_length[i]:>12.4f} {spread_by_length[i]:>9.4f}")

peak = int(np.argmax(mean_by_length)) + MIN_WORDS
drop = mean_by_length[peak - MIN_WORDS] - mean_by_length[-1]
print()
print(f"найкраща довжина : {peak} слів")
print(f"падіння від {peak} до {MAX_WORDS} слів: {drop:.4f}")
print(f"найбільший розкид по зернах у цьому відрізку: "
      f"{spread_by_length[peak - MIN_WORDS:].max():.4f}")

## 8 · А тепер — справжня генерація

Частка правильних токенів вище рахувалась **при поданому еталоні**. На кожному кроці
модель отримувала правильний початок перекладу й мусила назвати лише одне наступне
слово.

У бою еталона немає. Модель починає з `<bos>` і далі годує себе тим, що написала
сама. Одна помилка на другому кроці означає, що третій крок робиться вже з
неправильного місця.

Напишемо **жадібне декодування**: на кожному кроці беремо найімовірніше слово.

In [ ]:
MAX_OUT = 12          # запобіжник: далі декодер точно зациклився

def greedy_decode(model, sources, max_out=MAX_OUT, chunk_size=512):
    """Генерація без еталона. Повертає токени разом із <eos>, якщо модель його видала.
    Рахуємо шматками по 512 речень, щоб не тримати в памʼяті велетенську матрицю балів."""
    model.eval()
    outputs = []
    with torch.no_grad():
        for start in range(0, len(sources), chunk_size):
            batch = sources[start:start + chunk_size]
            src = pad_batch(batch)
            src_len = torch.tensor([len(s) for s in batch])
            hidden = model.context(src, src_len)
            current = torch.full((len(batch), 1), BOS, dtype=torch.long)
            rows = [[] for _ in batch]
            finished = [False] * len(batch)
            for _ in range(max_out):
                step, hidden = model.decoder(model.tgt_emb(current), hidden)
                chosen = model.out(step[:, 0, :]).argmax(-1)
                for i, token in enumerate(chosen.tolist()):
                    if finished[i]:
                        continue
                    rows[i].append(token)
                    if token == EOS:
                        finished[i] = True
                if all(finished):
                    break
                current = chosen.unsqueeze(1)
            outputs.extend(rows)
    return outputs

def words_only(sequence):
    """Прибираємо кінцевий <eos>, якщо він є: для друку й для BLEU потрібні слова."""
    return sequence[:-1] if sequence and sequence[-1] == EOS else sequence

valid_sources = [item[0] for item in valid_ids]
valid_refs_eos = [item[1][1:] for item in valid_ids]     # без <bos>, з <eos>
valid_refs = [words_only(ref) for ref in valid_refs_eos]  # самі слова

greedy_by_seed = [greedy_decode(model, valid_sources) for model in models]
greedy_words_by_seed = [[words_only(row) for row in g] for g in greedy_by_seed]
print("згенеровано перекладів:", len(greedy_by_seed[0]), "× 3 зерна")
print("приклад виходу першого зерна:", [tgt_itos[t] for t in greedy_words_by_seed[0][0]])
print("частка перекладів, де модель сама сказала <eos>:",
      f"{np.mean([row[-1] == EOS for row in greedy_by_seed[0]]):.4f}")

### Три мірки поруч

- **частка правильних токенів при еталоні** — те, що ми рахували вище;
- **частка правильних токенів у справжній генерації** — на тих самих позиціях,
  скільки разів згенероване слово збіглося з еталонним;
- **точний збіг** — скільки перекладів збіглися з еталоном **цілком**.

Третя мірка сувора до жорстокості, але саме її бачить читач.

In [ ]:
def free_token_accuracy(generated, references):
    """Скільки токенів збіглося з еталоном на тих самих позиціях."""
    hits = 0
    total = 0
    for candidate, reference in zip(generated, references):
        for i, word in enumerate(reference):
            total += 1
            if i < len(candidate) and candidate[i] == word:
                hits += 1
    return hits / total

def exact_match(generated, references):
    return sum(1 for c, r in zip(generated, references) if c == r) / len(references)

# порівнюємо з тим самим еталоном, що й при подаванні: разом із <eos>
free_acc = [free_token_accuracy(g, valid_refs_eos) for g in greedy_by_seed]
exact = [exact_match(g, valid_refs) for g in greedy_words_by_seed]

print(f"при поданому еталоні   : {np.mean(accuracies):.4f} ±{spread(accuracies):.4f}")
print(f"у справжній генерації  : {np.mean(free_acc):.4f} ±{spread(free_acc):.4f}")
print(f"точний збіг усього рядка: {np.mean(exact):.4f} ±{spread(exact):.4f}")
print()
print(f"генерація гірша за подавання еталона на "
      f"{np.mean(accuracies) - np.mean(free_acc):.4f}, "
      f"тобто у {np.mean(accuracies) / np.mean(free_acc):.2f} раза")

### Десять перекладів підряд

Не найкращих. Просто перших десять із перевірної частини — щоб було видно,
як воно є насправді.

In [ ]:
for i in range(10):
    source = ' '.join(src_itos[t] for t in valid_sources[i][:-1])     # без <eos>
    reference = ' '.join(tgt_itos[t] for t in valid_refs[i])
    machine = ' '.join(tgt_itos[t] for t in greedy_words_by_seed[0][i])
    print(f"{i+1}. англ    : {source}")
    print(f"   людина  : {reference}")
    print(f"   машина  : {machine}")
    print()

### Та сама крива, але без еталона

Розкладемо за довжиною джерела ще й справжню генерацію. Якщо горб — властивість
задачі, а не спосіб міряти, він має лишитись на місці.

In [ ]:
def free_accuracy_by_length(generated, references, lengths):
    """Частка правильних токенів у вільній генерації, окремо по довжині джерела."""
    hits = np.zeros(MAX_WORDS + 1)
    total = np.zeros(MAX_WORDS + 1)
    for candidate, reference, length in zip(generated, references, lengths):
        for i, word in enumerate(reference):
            total[length] += 1
            if i < len(candidate) and candidate[i] == word:
                hits[length] += 1
    return hits, total

source_lengths = [item[2] for item in valid_ids]
free_per_length = []
for generated in greedy_by_seed:
    hits, total = free_accuracy_by_length(generated, valid_refs_eos, source_lengths)
    free_per_length.append([ratio(hits[L], total[L]) for L in range(MIN_WORDS, MAX_WORDS + 1)])
free_per_length = np.array(free_per_length)
free_mean_by_length = free_per_length.mean(axis=0)
free_spread_by_length = (free_per_length.max(axis=0) - free_per_length.min(axis=0)) / 2

print(f"{'слів':>5} {'з еталоном':>12} {'розкид':>8} {'без еталона':>13} {'розкид':>8}")
for i, length in enumerate(range(MIN_WORDS, MAX_WORDS + 1)):
    print(f"{length:>5} {mean_by_length[i]:>12.4f} {spread_by_length[i]:>8.4f} "
          f"{free_mean_by_length[i]:>13.4f} {free_spread_by_length[i]:>8.4f}")
free_peak = int(np.argmax(free_mean_by_length)) + MIN_WORDS
print()
print(f"найкраща довжина без еталона: {free_peak} слів")
print(f"падіння від 4 до 8 слів без еталона: "
      f"{free_mean_by_length[2] - free_mean_by_length[-1]:.4f}")

## 9 · Де саме вона ламається: точність за позицією

Порівняємо два режими не в цілому, а **покроково**: наскільки часто вгадано перше
слово перекладу, друге, третє.

При поданому еталоні кожен крок починається з правильного місця, тож крива
має триматись рівно. У вільній генерації помилка накопичується.

In [ ]:
def accuracy_by_position(model, limit=8):
    """Частка правильних окремо для кожної позиції — у двох режимах."""
    forced_hits = np.zeros(limit); forced_total = np.zeros(limit)
    with torch.no_grad():
        for start in range(0, len(valid_ids), 256):
            chunk = valid_ids[start:start + 256]
            sources = [item[0] for item in chunk]
            src = pad_batch(sources)
            src_len = torch.tensor([len(s) for s in sources])
            tgt = pad_batch([item[1] for item in chunk])
            predicted = model(src, src_len, tgt[:, :-1]).argmax(-1)
            gold = tgt[:, 1:]
            # <eos> із підрахунку прибираємо: він не слово, і на останніх позиціях
            # його стає стільки, що крива міряла б уже не переклад, а вміння спинитись
            mask = (gold != PAD) & (gold != EOS)
            for position in range(min(limit, gold.shape[1])):
                forced_total[position] += mask[:, position].sum().item()
                forced_hits[position] += ((predicted[:, position] == gold[:, position])
                                          & mask[:, position]).sum().item()
    return forced_hits / np.maximum(forced_total, 1), forced_total

def free_accuracy_by_position(generated, references, limit=8):
    hits = np.zeros(limit); total = np.zeros(limit)
    for candidate, reference in zip(generated, references):
        for position in range(min(limit, len(reference))):
            if reference[position] == EOS:
                continue                     # той самий виняток, що й вище
            total[position] += 1
            if position < len(candidate) and candidate[position] == reference[position]:
                hits[position] += 1
    return hits / np.maximum(total, 1), total

forced_pos = np.array([accuracy_by_position(m)[0] for m in models])
free_pos = np.array([free_accuracy_by_position(g, valid_refs_eos)[0] for g in greedy_by_seed])
position_counts = accuracy_by_position(models[0])[1]

print(f"{'позиція':>8} {'токенів':>9} {'з еталоном':>12} {'без еталона':>13} {'різниця':>9}")
for position in range(8):
    if position_counts[position] < 50:
        continue
    a, b = forced_pos[:, position].mean(), free_pos[:, position].mean()
    print(f"{position+1:>8} {int(position_counts[position]):>9} "
          f"{a:>12.4f} {b:>13.4f} {a-b:>9.4f}")

## 10 · Жадібно чи променем: `beam search`

Жадібне декодування на кожному кроці бере найімовірніше слово й більше до нього
не повертається. Але найімовірніше слово зараз може закрити дорогу до кращого
продовження.

`beam search` тримає одразу `k` найкращих незавершених версій відповіді. На кожному
кроці кожну з них продовжують усіма кандидатами, а з усього, що вийшло, лишають
знову `k` найкращих. При `k = 1` це рівно жадібний вибір.

Оцінка версії — сума логарифмів імовірностей її слів. Логарифми додаються замість
того, щоб множити ймовірності: добуток вісьмох чисел, менших за одиницю, швидко
стає надто малим для комп'ютерної арифметики.

In [ ]:
def beam_decode_one(model, source, width, max_out=MAX_OUT, alpha=0.0):
    """beam search для одного речення.
    alpha — нормалізація довжини: 0 означає «не нормалізувати»."""
    src = torch.tensor([source])
    src_len = torch.tensor([len(source)])
    with torch.no_grad():
        hidden = model.context(src, src_len)
        # версія: (слова, сума логарифмів, стан GRU, чи завершена)
        beams = [([], 0.0, hidden, False)]
        for _ in range(max_out):
            if all(beam[3] for beam in beams):
                break
            candidates = []
            for words, score, state, finished in beams:
                if finished:
                    candidates.append((words, score, state, True))
                    continue
                previous = BOS if not words else words[-1]
                step, new_state = model.decoder(
                    model.tgt_emb(torch.tensor([[previous]])), state)
                log_probs = torch.log_softmax(model.out(step[:, 0, :]), -1)[0]
                best = torch.topk(log_probs, width)
                for delta, token in zip(best.values.tolist(), best.indices.tolist()):
                    if token == EOS:
                        candidates.append((words, score + delta, new_state, True))
                    else:
                        candidates.append((words + [token], score + delta, new_state, False))
            def rank(beam):
                length = max(len(beam[0]), 1)
                return beam[1] / (length ** alpha) if alpha else beam[1]
            candidates.sort(key=rank, reverse=True)
            beams = candidates[:width]
        winner = max(beams, key=lambda beam: (beam[1] / (max(len(beam[0]), 1) ** alpha))
                     if alpha else beam[1])
    return winner[0], winner[1]

# beam search дорогий: рахуємо його на підвибірці, зате однаковій для всіх ширин
BEAM_SAMPLE = 250
beam_sources = valid_sources[:BEAM_SAMPLE]
beam_refs = valid_refs[:BEAM_SAMPLE]
beam_refs_eos = valid_refs_eos[:BEAM_SAMPLE]
print("зразок для порівняння декодувань:", BEAM_SAMPLE, "речень")

In [ ]:
beam_results = {}
for width in (1, 2, 4, 8):
    started = time.process_time()
    generated = [beam_decode_one(models[0], source, width)[0] for source in beam_sources]
    seconds = time.process_time() - started
    beam_results[width] = dict(
        generated=generated,
        token=free_token_accuracy([g + [EOS] for g in generated], beam_refs_eos),
        exact=exact_match(generated, beam_refs),
        length=np.mean([len(g) for g in generated]),
        seconds=seconds)

reference_length = np.mean([len(r) for r in beam_refs])
print(f"{'ширина':>7} {'правильних':>12} {'точний збіг':>13} {'слів у виході':>15} {'час, с':>8}")
for width, result in beam_results.items():
    print(f"{width:>7} {result['token']:>12.4f} {result['exact']:>13.4f} "
          f"{result['length']:>15.2f} {result['seconds']:>8.1f}")
print(f"{'еталон':>7} {'':>12} {'':>13} {reference_length:>15.2f}")

### Чому промінь любить коротке

Сума логарифмів імовірностей завжди **спадає** з кожним новим словом: логарифм
числа, меншого за одиницю, відʼємний. Тому з двох версій — короткої й довгої —
проста сума майже завжди обирає коротку, навіть якщо довга краща.

Лікується діленням на довжину в степені `alpha`. Перевіримо, чи це щось міняє.

In [ ]:
for alpha in (0.0, 0.7, 1.0):
    generated = [beam_decode_one(models[0], source, 4, alpha=alpha)[0]
                 for source in beam_sources]
    print(f"alpha={alpha:.1f}: слів у виході {np.mean([len(g) for g in generated]):.2f}, "
          f"правильних {free_token_accuracy([g + [EOS] for g in generated], beam_refs_eos):.4f}, "
          f"точний збіг {exact_match(generated, beam_refs):.4f}")
print(f"еталон   : слів у виході {reference_length:.2f}")

### Промені зсередини — на одному реченні

Візьмемо перше речення перевірної частини й подивимось, що саме тримає промінь
ширини 4 на кожному кроці. Ці числа малює перша інтерактивна фігура лекції.

In [ ]:
def beam_trace(model, source, width, max_out=6):
    """Ті самі кроки, але з журналом: які версії лишились після кожного кроку."""
    src = torch.tensor([source]); src_len = torch.tensor([len(source)])
    trace = []
    with torch.no_grad():
        hidden = model.context(src, src_len)
        beams = [([], 0.0, hidden, False)]
        for _ in range(max_out):
            candidates = []
            for words, score, state, finished in beams:
                if finished:
                    candidates.append((words, score, state, True)); continue
                previous = BOS if not words else words[-1]
                step, new_state = model.decoder(model.tgt_emb(torch.tensor([[previous]])), state)
                log_probs = torch.log_softmax(model.out(step[:, 0, :]), -1)[0]
                best = torch.topk(log_probs, width)
                for delta, token in zip(best.values.tolist(), best.indices.tolist()):
                    if token == EOS:
                        candidates.append((words, score + delta, new_state, True))
                    else:
                        candidates.append((words + [token], score + delta, new_state, False))
            candidates.sort(key=lambda beam: beam[1], reverse=True)
            beams = candidates[:width]
            trace.append([(' '.join(tgt_itos[t] for t in b[0]) or '·',
                           round(b[1], 3), b[3]) for b in beams])
            if all(b[3] for b in beams):
                break
    return trace

showcase = 0
print("джерело :", ' '.join(src_itos[t] for t in valid_sources[showcase][:-1]))
print("еталон  :", ' '.join(tgt_itos[t] for t in valid_refs[showcase]))
for width in (1, 2, 3, 4):
    print()
    print(f"--- ширина променя {width} ---")
    for step, level in enumerate(beam_trace(models[0], valid_sources[showcase], width), 1):
        print(f"крок {step}:")
        for text, score, finished in level:
            print(f"   {score:>8.3f}  {text}{'  ✔' if finished else ''}")

### Розподіл на кожному кроці

І те саме жадібним поглядом: п'ять найімовірніших слів на кожному кроці для того
самого речення. Видно, наскільки впевнена модель.

In [ ]:
def top5_trace(model, source, max_out=6):
    src = torch.tensor([source]); src_len = torch.tensor([len(source)])
    rows = []
    with torch.no_grad():
        hidden = model.context(src, src_len)
        current = torch.tensor([[BOS]])
        for _ in range(max_out):
            step, hidden = model.decoder(model.tgt_emb(current), hidden)
            probs = torch.softmax(model.out(step[:, 0, :]), -1)[0]
            best = torch.topk(probs, 5)
            rows.append([(tgt_itos[t], round(p, 4))
                         for p, t in zip(best.values.tolist(), best.indices.tolist())])
            chosen = int(best.indices[0])
            if chosen == EOS:
                break
            current = torch.tensor([[chosen]])
    return rows

for step, row in enumerate(top5_trace(models[0], valid_sources[showcase]), 1):
    print(f"крок {step}: " + "  ".join(f"{word}={p:.3f}" for word, p in row))

## 11 · BLEU: як міряють переклад — і чому тут він майже мовчить

`BLEU` дивиться, яка частка n-грам машинного перекладу трапляється в людському.
Рахують чотири точності — за одинарними словами, парами, трійками й четвірками, —
беруть їхнє середнє геометричне й множать на **штраф за стислість**: інакше
найвигіднішою стратегією було б видавати одне-єдине дуже ймовірне слово.

`sacrebleu` у середовищі немає, тож пишемо самі. І одразу звіряємо з `nltk`.

In [ ]:
def corpus_bleu_ours(candidates, references, max_n=4, nltk_quirk=False):
    """Власна реалізація BLEU на рівні корпусу.
    Повертає сам BLEU, чотири точності, штраф за стислість і дві довжини.
    nltk_quirk відтворює одну деталь реалізації nltk — про неї нижче."""
    matched = [0] * max_n
    produced = [0] * max_n
    candidate_length = 0
    reference_length = 0
    for candidate, reference in zip(candidates, references):
        candidate_length += len(candidate)
        reference_length += len(reference)
        for n in range(1, max_n + 1):
            in_candidate = collections.Counter(
                tuple(candidate[i:i + n]) for i in range(len(candidate) - n + 1))
            in_reference = collections.Counter(
                tuple(reference[i:i + n]) for i in range(len(reference) - n + 1))
            total_here = sum(in_candidate.values())
            # nltk боїться ділення на нуль і ставить одиницю там, де n-грам немає
            produced[n - 1] += max(1, total_here) if nltk_quirk else total_here
            # кожну n-граму зараховуємо не більше разів, ніж вона є в еталоні
            matched[n - 1] += sum(min(count, in_reference[gram])
                                  for gram, count in in_candidate.items())
    precisions = [matched[i] / produced[i] if produced[i] else 0.0 for i in range(max_n)]
    if min(precisions) == 0:
        geometric = 0.0
    else:
        geometric = float(np.exp(sum(np.log(p) for p in precisions) / max_n))
    if candidate_length == 0:
        brevity = 0.0
    elif candidate_length > reference_length:
        brevity = 1.0
    else:
        brevity = float(np.exp(1 - reference_length / candidate_length))
    return brevity * geometric, precisions, brevity, candidate_length, reference_length

from nltk.translate.bleu_score import corpus_bleu as nltk_bleu

greedy_words = [[tgt_itos[t] for t in row] for row in greedy_words_by_seed[0]]
reference_words = [[tgt_itos[t] for t in row] for row in valid_refs]

ours, precisions, brevity, c_len, r_len = corpus_bleu_ours(greedy_words, reference_words)
same_as_nltk = corpus_bleu_ours(greedy_words, reference_words, nltk_quirk=True)[0]
theirs = nltk_bleu([[ref] for ref in reference_words], greedy_words)

print(f"наш BLEU, чесний знаменник : {ours:.6f}")
print(f"наш BLEU, як рахує nltk    : {same_as_nltk:.6f}")
print(f"nltk BLEU                  : {theirs:.6f}")
assert abs(same_as_nltk - theirs) < 1e-9, "розрахунок розійшовся!"
print("✅ збігається з nltk до знака")
print()
print("Різниця між двома нашими числами —", f"{ours - same_as_nltk:.6f}", "— це не помилка.")
print("Речення, коротше за чотири слова, не має жодної четвірки слів. Ділити нема на що,")
print("і nltk кладе в знаменник одиницю, щоб не ділити на нуль. Наш корпус короткий,")
print("таких речень багато — і деталь реалізації зсуває метрику на видимому знаку.")
print()
for i, p in enumerate(precisions, 1):
    print(f"  точність за {i}-грамами: {p:.4f}")
print(f"  штраф за стислість      : {brevity:.4f}")
print(f"  слів машиною / людиною  : {c_len} / {r_len}")

In [ ]:
# BLEU для всіх зерен і для променя — на тій самій підвибірці, що й вище
beam_words = {w: [[tgt_itos[t] for t in row] for row in r['generated']]
              for w, r in beam_results.items()}
sample_refs = reference_words[:BEAM_SAMPLE]

bleu_greedy_seeds = [corpus_bleu_ours([[tgt_itos[t] for t in row] for row in g],
                                      reference_words)[0] for g in greedy_words_by_seed]
print(f"BLEU жадібно, уся перевірна частина: "
      f"{np.mean(bleu_greedy_seeds):.4f} ±{spread(bleu_greedy_seeds):.4f}")
print()
print(f"{'ширина':>7} {'BLEU-4':>9} {'BLEU-2':>9}")
for width in beam_words:
    print(f"{width:>7} {corpus_bleu_ours(beam_words[width], sample_refs)[0]:>9.4f} "
          f"{corpus_bleu_ours(beam_words[width], sample_refs, max_n=2)[0]:>9.4f}")
print()
print("BLEU-2 наведено поруч не для краси. Середнє геометричне обертається на нуль,")
print("щойно хоч одна з чотирьох точностей дорівнює нулю, — а на короткій вибірці")
print("з коротких речень цілком можлива відсутність бодай однієї збіглої четвірки слів.")
print("Тоді BLEU-4 скаже «нуль» про переклад, у якому збіглася половина слів.")

### Чому BLEU суворий саме до української

Візьмемо один еталонний переклад і зіпсуємо його трьома різними способами:
поставимо слово в інший відмінок, переставимо два слова місцями й замінимо слово
синонімом. Людина назве всі три варіанти правильними. Подивимось на BLEU.

Це **зроблений руками** приклад, а не замір на корпусі: він показує механізм
метрики, а не якість нашої моделі.

In [ ]:
reference_demo = "не вдалося зберегти поточний документ".split()
variants = {
    "точна копія":        "не вдалося зберегти поточний документ".split(),
    "інший відмінок":     "не вдалося зберегти поточного документа".split(),
    "переставлені слова": "не вдалося поточний документ зберегти".split(),
    "синонім":            "не вдалося записати поточний документ".split(),
    "утричі коротше":     "не вдалося".split(),
}
print(f"{'варіант':<20} {'BLEU':>8} {'1-грами':>9} {'2-грами':>9} "
      f"{'3-грами':>9} {'4-грами':>9} {'штраф':>8} {'слів':>6}")
for name, variant in variants.items():
    value, precisions_demo, brevity_demo, _, _ = corpus_bleu_ours([variant], [reference_demo])
    print(f"{name:<20} {value:>8.4f} {precisions_demo[0]:>9.4f} "
          f"{precisions_demo[1]:>9.4f} {precisions_demo[2]:>9.4f} "
          f"{precisions_demo[3]:>9.4f} {brevity_demo:>8.4f} {len(variant):>6d}")

## 12 · Чи справді винен розмір вектора контексту

Крива за довжиною падає. Гіпотеза: винен вектор контексту — у нього завжди `H`
чисел, скільки б слів не було в реченні, тож на кожне слово довгого речення
припадає менше місця.

Гіпотезу можна перевірити прямо: навчимо ту саму модель на тих самих даних,
але з удвічі-втричі коротшим вектором. Якщо справа в ньому, довгі речення мають
постраждати **сильніше** за короткі.

In [ ]:
SMALL_HID = 64

small_models, small_times, small_acc, small_per_length = [], [], [], []
for seed in SEEDS:
    model, seconds, loss = train_model(seed, hid=SMALL_HID)
    correct, total = token_accuracy(model)
    small_models.append(model)
    small_times.append(seconds)
    small_acc.append(correct.sum() / total.sum())
    small_per_length.append([ratio(correct[L], total[L]) for L in range(MIN_WORDS, MAX_WORDS + 1)])
    print(f"зерно {seed}: H={SMALL_HID}, частка правильних {small_acc[-1]:.4f} · "
          f"навчання {seconds:.1f} с")

small_per_length = np.array(small_per_length)
small_mean = small_per_length.mean(axis=0)
small_spread = (small_per_length.max(axis=0) - small_per_length.min(axis=0)) / 2
print()
print(f"РАЗОМ H={SMALL_HID}: {np.mean(small_acc):.4f} ±{spread(small_acc):.4f} · "
      f"{np.mean(small_times):.1f} ±{spread(small_times):.1f} с")

In [ ]:
print(f"H={SMALL_HID} по зернах:")
for j, seed in enumerate(SEEDS):
    print(f"  зерно {seed}: " + ' '.join(f'{v:.4f}' for v in small_per_length[j]))
print()
print(f"{'слів':>5} {'H='+str(HID):>10} {'розкид':>8} {'H='+str(SMALL_HID):>10} {'розкид':>8} {'різниця':>9}")
for i, length in enumerate(range(MIN_WORDS, MAX_WORDS + 1)):
    print(f"{length:>5} {mean_by_length[i]:>10.4f} {spread_by_length[i]:>8.4f} "
          f"{small_mean[i]:>10.4f} {small_spread[i]:>8.4f} "
          f"{mean_by_length[i]-small_mean[i]:>9.4f}")

big_drop = mean_by_length[2] - mean_by_length[-1]        # від 4 слів до 8
small_drop = small_mean[2] - small_mean[-1]
print()
print(f"падіння від 4 до 8 слів, H={HID}: {big_drop:.4f}")
print(f"падіння від 4 до 8 слів, H={SMALL_HID}: {small_drop:.4f}")
print(f"різниця падінь: {small_drop - big_drop:+.4f}")
print(f"розкид падіння по зернах, H={HID}: "
      f"{spread(list(per_length[:, 2] - per_length[:, -1])):.4f}")
print(f"розкид падіння по зернах, H={SMALL_HID}: "
      f"{spread(list(small_per_length[:, 2] - small_per_length[:, -1])):.4f}")

### Скільки чисел вектора припадає на слово

Проста арифметика, яка стоїть за гіпотезою: вектор має сталу довжину, а слів
буває різна кількість.

In [ ]:
print(f"{'слів':>5} {'H='+str(HID)+' на слово':>16} {'H='+str(SMALL_HID)+' на слово':>16} {'правильних, H='+str(HID):>20}")
for i, length in enumerate(range(MIN_WORDS, MAX_WORDS + 1)):
    print(f"{length:>5} {HID/length:>16.1f} {SMALL_HID/length:>16.1f} {mean_by_length[i]:>20.4f}")

## 13 · Числа, які цитує лекція

Одним місцем — щоб їх можна було звірити не читаючи весь зошит.

In [ ]:
print("КОРПУС")
print(f"  пар 2-8 слів              : {len(pairs)}")
print(f"  навчальних / перевірних   : {len(train)} / {len(valid)}")
print(f"  словники англ / укр       : {len(src_itos)} / {len(tgt_itos)}")
print(f"  <unk> у перевірній цілі   : {unknown_share:.4f}")
print()
print("МОДЕЛЬ")
print(f"  EMB={EMB} HID={HID} батч={BATCH} епох={EPOCHS} lr={LR}")
print(f"  параметрів                : {params}")
print()
print("ЯКІСТЬ")
print(f"  при поданому еталоні      : {np.mean(accuracies):.4f} ±{spread(accuracies):.4f}")
print(f"  вільна генерація, токени  : {np.mean(free_acc):.4f} ±{spread(free_acc):.4f}")
print(f"  точний збіг рядка         : {np.mean(exact):.4f} ±{spread(exact):.4f}")
print(f"  BLEU жадібно              : {np.mean(bleu_greedy_seeds):.4f} ±{spread(bleu_greedy_seeds):.4f}")
print(f"  навчання на зерно         : {np.mean(train_times):.1f} ±{spread(train_times):.1f} с")
print()
print("ЗА ДОВЖИНОЮ ДЖЕРЕЛА (при поданому еталоні)")
print("  H=%d : %s" % (HID, ' '.join(f'{v:.4f}' for v in mean_by_length)))
print("  розкид: %s" % ' '.join(f'{v:.4f}' for v in spread_by_length))
print("  H=%d : %s" % (SMALL_HID, ' '.join(f'{v:.4f}' for v in small_mean)))
print("  розкид: %s" % ' '.join(f'{v:.4f}' for v in small_spread))
print("  без еталона: %s" % ' '.join(f'{v:.4f}' for v in free_mean_by_length))
print("  розкид     : %s" % ' '.join(f'{v:.4f}' for v in free_spread_by_length))
print()
print("ЗА ПОЗИЦІЄЮ В ПЕРЕКЛАДІ")
print("  з еталоном : %s" % ' '.join(f'{v:.4f}' for v in forced_pos.mean(axis=0)[:6]))
print("  без еталона: %s" % ' '.join(f'{v:.4f}' for v in free_pos.mean(axis=0)[:6]))
print()
print("ДЕКОДУВАННЯ (зерно 0, %d речень)" % BEAM_SAMPLE)
for width, result in beam_results.items():
    print(f"  ширина {width}: токенів {result['token']:.4f}, точний збіг {result['exact']:.4f}, "
          f"слів {result['length']:.2f}, BLEU {corpus_bleu_ours(beam_words[width], sample_refs)[0]:.4f}")
print()
print(f"ЧАС ЗОШИТА: {time.process_time() - NOTEBOOK_START:.1f} с процесорних")

## Завдання

### 🟢 Рівень 1 — База

Заміни жадібне декодування на `beam search` ширини 2 в клітинці, де рахується
частка правильних токенів у справжній генерації, і назви обидва числа.

**Зроблено, якщо:** названо частку правильних для обох способів і сказано,
чи різниця більша за розкид по трьох зернах.

### 🟡 Рівень 2 — Плюс

Навчи модель у зворотний бік: з української англійською. Словники міняються
місцями, решта коду та сама.

**Зроблено, якщо:** названо частку правильних в обидва боки й пояснено,
чому один напрямок легший.

### 🔴 Рівень 3 — Виклик

Подавай декодерові вектор контексту **на кожному кроці**, а не лише як стартовий
стан: додай його до ембедингу слова перед входом у GRU.

**Зроблено, якщо:** побудовано криву за довжиною джерела для обох варіантів
і сказано, чи змінився нахил від 4 до 8 слів більше за розкид по зернах.